# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution — Exploration with `mlcroissant`

This notebook demonstrates how to load and explore a FAIR² Croissant dataset using the [mlcroissant](https://mlcroissant.readthedocs.io/en/stable/) library. The workflow includes loading the dataset metadata, examining its schema, extracting record sets, performing exploratory analysis, and basic visualization—all driven by Croissant `@id` referencing for full schema transparency.

### Dataset Source
This dataset is described via a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load dataset metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's examine the available record sets, their fields, and the associated `@id`s from the Croissant schema.

In [ ]:
# List available record sets (`@id` and name)
record_sets = []
print("Available Record Sets:")
for rs in dataset.record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")
    record_sets.append(rs.id)
    # List fields for each record set
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print()

# Save first record set's id for later use
if record_sets:
    primary_record_set_id = record_sets[0]
else:
    raise ValueError("No record sets found in this dataset.")

## 3. Data Extraction

Load data from each record set into a DataFrame. This uses each record set's `@id` as reference for clarity and reproducibility.

In [ ]:
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Shape: {df.shape}\n")
    else:
        print("  No records found for this record set!\n")

# Review the first few rows of the main record set
print(f"\nPreview of primary record set (@id: {primary_record_set_id}):")
display_cols = dataframes[primary_record_set_id].columns.tolist() if primary_record_set_id in dataframes else []
print(display_cols)
if primary_record_set_id in dataframes:
    display_df = dataframes[primary_record_set_id]
    display_df.head()

## 4. Exploratory Data Analysis (EDA)

In this section, we'll select a numerical field (by `@id`), filter records (e.g., by age), normalize a field, and explore groupwise aggregations, using only Croissant-defined `@id`s for reference.

In [ ]:
# Get the main DataFrame and its field definitions via Croissant
df = dataframes[primary_record_set_id]
fields = {f.id: f for f in next(rs for rs in dataset.record_sets if rs.id == primary_record_set_id).fields}

# Show fields with their types
print("Numeric fields detected in main record set:")
numeric_field_ids = [fid for fid, f in fields.items() if f.data_type in ("Float", "Integer", "Number") and fid in df.columns]
for fid in numeric_field_ids:
    print(f"- {fid}: {fields[fid].name}")

# Select a numeric field by its @id — adjust depending on whats available:
if numeric_field_ids:
    numeric_field_id = numeric_field_ids[0]
else:
    raise ValueError("No numeric field found to analyze.")

# Filter records with this numeric field above a threshold
threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'iufc' else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Find a likely categorical field for grouping (non-numeric)
group_field_candidates = [fid for fid in fields if fid != numeric_field_id and fid in df.columns and df[fid].dtype == object]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    group_field_id = None
    print("No categorical grouping field found.")

## 5. Visualization

Visualize key distributions in the main tabular record set, with field references by their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f"Distribution of {fields[numeric_field_id].name} (@id: {numeric_field_id})")
plt.xlabel(fields[numeric_field_id].name)
plt.ylabel('Count')
plt.show()

# If a grouping field was found, show boxplot
if group_field_id:
    plt.figure(figsize=(9,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{fields[numeric_field_id].name} by {fields[group_field_id].name} (@id: {group_field_id})")
    plt.xlabel(fields[group_field_id].name)
    plt.ylabel(fields[numeric_field_id].name)
    plt.xticks(rotation=30, ha='right')
    plt.show()

## 6. Conclusion

- We demonstrated schema-driven loading and inspection of a FAIR² tabular dataset using the `mlcroissant` library, referencing all data by their Croissant `@id`s.
- Data fields, including numeric and categorical variables, were discovered dynamically from the Croissant schema. 
- Simple filtering, normalization, grouping, and plotting was performed without hard-coding field names, thus making the process robust and reusable for schema-defined datasets.

For deeper clinical and statistical analyses, further field-level annotations can be leveraged from the schema, and the same workflow can be extended to other Croissant datasets.